# 00 — Setup e Validação da Conexão BD+

**Objetivo:** confirmar que o ambiente está configurado corretamente e que a conexão com o BigQuery via Base dos Dados funciona.

**Pré-requisitos:**
- Arquivo `.env` preenchido com `GCP_PROJECT_ID` e `GOOGLE_APPLICATION_CREDENTIALS`
- Arquivo `gcp-service-account.json` na raiz do projeto

**Output esperado (última célula):** `✅ Conexão validada — 22 municípios do Acre encontrados`

In [ ]:
import os
from dotenv import load_dotenv
import basedosdados as bd
import pandas as pd

load_dotenv()

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = os.getenv("GOOGLE_APPLICATION_CREDENTIALS", "")
PROJECT_ID = os.getenv("GCP_PROJECT_ID")

print(f"Projeto GCP  : {PROJECT_ID}")
print(f"Credenciais  : {os.environ['GOOGLE_APPLICATION_CREDENTIALS']}")
print(f"Arquivo existe: {os.path.exists(os.environ['GOOGLE_APPLICATION_CREDENTIALS'])}")

In [ ]:
# Teste de conexão — buscar os 5 maiores municípios do Acre
df_teste = bd.read_sql(
    query="""
        SELECT id_municipio, nome, sigla_uf, populacao
        FROM `basedosdados.br_ibge_populacao.municipio`
        WHERE sigla_uf = 'AC' AND ano = 2022
        ORDER BY populacao DESC
        LIMIT 5
    """,
    billing_project_id=PROJECT_ID
)
df_teste

In [ ]:
# Validação: o Acre deve ter exatamente 22 municípios
df_total = bd.read_sql("""
    SELECT COUNT(DISTINCT id_municipio) AS total
    FROM `basedosdados.br_ibge_populacao.municipio`
    WHERE sigla_uf = 'AC' AND ano = 2022
""", billing_project_id=PROJECT_ID)

total = df_total["total"].iloc[0]
assert total == 22, f"Esperado 22 municípios no AC, encontrado {total}"

# Validação: todos os IDs devem começar com '12' (código IBGE do Acre)
df_ids = bd.read_sql("""
    SELECT DISTINCT id_municipio
    FROM `basedosdados.br_ibge_populacao.municipio`
    WHERE sigla_uf = 'AC' AND ano = 2022
""", billing_project_id=PROJECT_ID)

assert all(df_ids["id_municipio"].astype(str).str.startswith("12")), \
    "IDs fora do padrão IBGE para o Acre"

print("✅ Conexão validada — 22 municípios do Acre encontrados")